# Identification of a 2-DOF Mass-Spring-Damper System with the RFP Method

***

This notebook demonstrates the identification of a 2-DOF mass-spring-damper system using the Rational Fraction Polynomial (RFP) method, presented for the first time by [Richardson & Formenti (1982)](http://papers.vibetech.com/Paper07.pdf). Unlike the LSCE algorithm explored in our previous notebooks, which works in the time domain by curve fitting the impulse response function (IRF), the RFP method operates in the frequency domain by curve fitting directly the frequency response function (FRF). This approach avoids the issues related to taking the inverse Fourier transform of FRFs that are inevitably limited in their frequency range, which cause a potentially serious error called _time domain leakage_. In fact, looking at the IRFs obtained in [notebook 2](02_Identification_of_a_1-DOF_Mass-Spring-Damper_Sytem_with_the_LSCE_Algorithm.ipynb), it is always possible to notice artificial oscillations at the end of the response, even in the IRF obtained from the Savitzky-Golay-filtered FRF, which produces the best results in terms of modal parameters. By working in the frequency domain, the RFP method is able to avoid this issue and provide a more robust estimation of the system's modal parameters.

To understand the RFP method and its application, we'll follow a structured approach with these key steps:

* [Theoretical Background](#theoretical-background)
    * [Modeling System Dynamics in the Laplace Domain](#modeling-laplace)
    * [Transfer Function and FRF of the 2-DOF System](#transfer-function-frf)
    * [RFP Method](#rfp-method)
    * [Orthogonal Polynomials Implementation](#orthogonal-polynomials)
    * [Global Parameter Estimation](#global-estimation)
* [Parameter Estimation from Random Excitation](#estimation-random)
    * [Data Generation](#data-generation)
    * [Data Processing](#data-processing)
    * [Parameter Estimation](#parameter-estimation)
* [Conclusions](#conclusions)

## Theoretical Background <a name="theoretical-background"></a>

***

To introduce the RFP method, we'll first review the mathematical representation of our 2-DOF system in the Laplace domain, followed by the derivation of its transfer function and FRF. This introduction will allow us to understand how the RFP representation of the FRF emerges naturally from the physics of the system. Successively, we'll see how the modal parameters can be extracted from the RFP representation. Finally, we'll discuss the need for orthogonal polynomials to avoid numerical issues in the curve-fitting process and the global parameter estimation strategy.

### Modeling System Dynamics in the Laplace Domain <a name="laplace-domain"></a>

The dynamics of mechanical structures can be efficiently represented in the Laplace domain, where time derivatives are replaced by algebraic operations. This transformation converts the system's differential equations of motion into algebraic equations, greatly simplifying the analysis of dynamic behavior.

In the Laplace domain, a system is characterized by its transfer functions, which relate the Laplace-transformed inputs to the Laplace-transformed outputs. These transfer functions are complex-valued functions of the complex Laplace variable $s$,   and contain all the information needed to describe the system's response to arbitrary inputs.

Let's consider the same 2-DOF mass-spring-damper system that we analyzed in our last notebook.

![2-DOF Mass-Spring-Damper System](resources/03_2DofMassSpringDamper.svg)

The equations of motion in the time domain are:

For mass $m_1$:
$$m_1\ddot{x}_1 + c_2(\dot{x}_1 - \dot{x}_2) + c_1\dot{x}_1 + k_2(x_1 - x_2) + k_1x_1 = 0$$

For mass $m_2$:
$$m_2\ddot{x}_2 + c_2(\dot{x}_2 - \dot{x}_1) + k_2(x_2 - x_1) = F(t)$$

Taking the Laplace transform of these equations with zero initial conditions yields:

$$m_1s^2X_1(s) + c_2s(X_1(s) - X_2(s)) + c_1sX_1(s) + k_2(X_1(s) - X_2(s)) + k_1X_1(s) = 0$$

$$m_2s^2X_2(s) + c_2s(X_2(s) - X_1(s)) + k_2(X_2(s) - X_1(s)) = F(s)$$

where $X_1(s)$, $X_2(s)$, and $F(s)$ are the Laplace transforms of $x_1(t)$, $x_2(t)$, and $F(t)$, respectively. In matrix form, these equations can be written as:

$$\mathbf{X}(s) \left[s^2\mathbf{M} + s\mathbf{C} + \mathbf{K}\right] = \mathbf{F}(s)$$

where $\mathbf{M}$, $\mathbf{C}$, and $\mathbf{K}$ are the mass, damping, and stiffness matrices, respectively, and $\mathbf{X}(s)$ and $\mathbf{F}(s)$ are the displacement and force vectors in the Laplace domain. The transfer function matrix $\mathbf{H}(s)$ of the system can be exposed by rearraging the above equation to:

$$\mathbf{X}(s) = \left[s^2\mathbf{M} + s\mathbf{C} + \mathbf{K}\right]^{-1}\mathbf{F}(s) = \mathbf{H}(s)\mathbf{F}(s).$$

In general, the transfer function matrix $\mathbf{H}(s)$ contains the transfer functions between each possible input-output DOF pair of the system. Since in our case we only have a single input force applied to the second mass, we are only interested in the transfer function matrix components $H_{12}(s)$ and $H_{22}(s)$, which relate the input force at $m_2$ to the displacement responses at $m_1$ and $m_2$, respectively.

### Transfer Function and FRF of the 2-DOF System <a name="transfer-function-frf"></a>

For our 2-DOF system, we can find the analytical expressions of $H_{12}(s)$ and $H_{22}(s)$. In fact, the transfer function matrix is given by:

$$\mathbf{H}(s) = \begin{bmatrix} H_{11}(s) & H_{12}(s) \\ H_{21}(s) & H_{22}(s) \end{bmatrix} = \left[s^2\mathbf{M} + s\mathbf{C} + \mathbf{K}\right]^{-1} = \begin{bmatrix} m_1s^2 + (c_1 + c_2)s + (k_1 + k_2) & -c_2s - k_2 \\ -c_2s - k_2 & m_2s^2 + c_2s + k_2 \end{bmatrix}^{-1},$$

and the transfer functions $H_{12}(s)$ and $H_{22}(s)$ can be derived using the formula for the inverse of a 2x2 matrix:

$$\begin{bmatrix} a & b \\ c & d \end{bmatrix}^{-1} = \frac{1}{ad - bc} \begin{bmatrix} d & -b \\ -c & a \end{bmatrix}.$$

Thus, the transfer functions $H_{12}(s)$ and $H_{22}(s)$ are:

$$H_{12}(s) = \frac{-(-c_2s - k_2)}{D(s)}$$

$$H_{22}(s) = \frac{m_1s^2 + (c_1 + c_2)s + (k_1 + k_2)}{D(s)},$$

where $D(s)$ is the determinant of the system matrix:

$$\begin{aligned}
D(s) &= \left(m_1s^2 + (c_1 + c_2)s + (k_1 + k_2)\right)(m_2s^2 + c_2s + k_2) - (c_2s + k_2)^2 = \\
&= m_1m_2s^4 + \left(m_1c_2 + m_2(c_1 + c_2)\right)s^3 + \left(m_1k_2 + m_2(k_1 + k_2) + c_1c_2\right)s^2 + \left(c_1k_2 + c_2(k_1 + k_2)\right)s + k_1k_2.
\end{aligned}$$

Let's now take a moment to examine the obtained expressions. The transfer functions $H_{12}(s)$ and $H_{22}(s)$ are rational functions of $s$, with different numerator polynomials of order 1 and 2, respectively, and a common denominator polynomial of order 4. The lower order of the numerator polynomial of $H_{12}(s)$ reflects the fact that the force at mass 2 must "travel" through the spring-damper connection to affect mass 1, while it directly affects mass 2. The order 4 of the denominator polynomial reflects the 2-DOF nature of the system, with both masses' inertial effects contributing to the overall dynamics.

We can observe that the magnitude of the transfer functions goes to zero when the numerator polynomial is zero, while it goes to infinity when the denominator polynomial is zero. The values of $s$ for which the numerator polynomial is zero are called the _zeros_ of the transfer function, while the values for which the denominator polynomial is zero are called the _poles_. The poles define the system's resonant conditions, where the system amplifies the input force.

The complex Laplace variable $s$ can be written in terms of its real and imaginary parts $s = \sigma + i\omega$, where $\sigma$ corresponds to a damping term and $\omega$ to a frequency term. This means that the Laplace variable defines a complex $s$-plane, with a damping and a frequency axis. The FRF is obtained by evaluating the transfer function along the frequency axis, or in other words at $s = i\omega$. This substitution transforms the Laplace domain representation to the frequency domain, allowing us to analyze the system's steady-state response to sinusoidal inputs.

For convenience, we can indicate the transfer functions and FRFs of our 2-DOF system simply as $H_1$ and $H_2$. Evaluating the transfer functions at $s = i\omega$ yields the FRFs:

$$H_1(i\omega) = \frac{(c_2i\omega + k_2)}{D(i\omega)}$$
$$H_2(i\omega) = \frac{(m_1i\omega^2 + (c_1 + c_2)i\omega + (k_1 + k_2))}{D(i\omega)},$$

where the denominator is now:

$$D(i\omega) = m_1m_2(i\omega)^4 + [m_1c_2 + m_2(c_1+c_2)](i\omega)^3 + [m_1k_2 + m_2(k_1+k_2) + c_1c_2](i\omega)^2 + [c_1k_2 + c_2(k_1+k_2)]i\omega + k_1k_2$$

As we know from our previous notebooks, the FRF is a complex-valued function of frequency. Its magnitude represents the amplitude ratio between output and input at each frequency, while its phase represents the phase shift between output and input. Compared to the transfer functions, the FRFs maintain the rational function form, with the same order of the numerator and denominator polynomials.

### RFP Method <a name="rfp-method"></a>

As we have seen, the FRFs of our 2-DOF system naturally takes the form of a ratio of polynomials in $i\omega$. This form is the foundation of the RFP method, which recognize that the FRF of a vibrating system can be mathematically represented as the ratio of two polynomials:

$$H(\omega) = \frac{\sum_{k=0}^{m} a_k s^k}{\sum_{k=0}^{n} b_k s^k}\Biggr|_{s=i\omega}$$

where $a_k$ and $b_k$ are the coefficients to be determined, and $m$ and $n$ are the orders of the numerator and denominator polynomials, respectively. Note that for a proper physical system, we must always have $m \leq n$.

The RFP method involves fitting measured FRF data to this rational function form to determine the coefficients $a_k$ and $b_k$. Once these coefficients are known, the poles and residues can be extracted, yielding the modal parameters and the mode shapes of the system.

To find the coefficients $a_k$ and $b_k$ that best fit the measured FRF data, the RFP method aims at minimizing the error between the above analytical expression and the measured FRF data. If we rearrange the above equation to separate the $a_k$ and $b_k$ terms, we get:

$$\sum_{k=0}^{m} a_k (i\omega)^k - H(\omega)\sum_{k=0}^{n} b_k (i\omega)^k = 0,$$

we can define the error at the $j$-th frequency point $\omega_j$ as:

$$e_j = \sum_{k=0}^{m} a_k (i\omega_j)^k - H_j\sum_{k=0}^{n} b_k (i\omega_j)^k,$$

where $H_j$ is the measured FRF value at frequency $\omega_j$.

We assume the highest order denominator coefficient to be unity, $b_n = 1$, which allows us to rewrite the error as: 

$$e_j = \sum_{k=0}^{m} a_k (i\omega_i)^k - H_j\left(\sum_{k=0}^{n-1} b_k (i\omega_j)^k + (i\omega_j)^{n}\right).$$

Considering the error vector $\mathbf{E}$ as the collection of all errors at the $L$ measured frequency points, we can obtain the following expression for the error vector:

$$\mathbf{E} = \mathbf{P}\mathbf{A} - \mathbf{T}\mathbf{B} - \mathbf{W},$$

where:

$$\mathbf{P} = \begin{bmatrix} 1 & i\omega_1 & (i\omega_1)^2 & \ldots & (i\omega_1)^m \\ 1 & i\omega_2 & (i\omega_2)^2 & \ldots & (i\omega_2)^m \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & i\omega_L & (i\omega_L)^2 & \ldots & (i\omega_L)^m \end{bmatrix},$$

$$\mathbf{T} = \begin{bmatrix} H_1 & H_1(i\omega_1) & H_1(i\omega_1)^2 & \ldots & H_1(i\omega_1)^{n-1} \\ H_2 & H_2(i\omega_2) & H_2(i\omega_2)^2 & \ldots & H_2(i\omega_2)^{n-1} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ H_L & H_L(i\omega_L) & H_L(i\omega_L)^2 & \ldots & H_L(i\omega_L)^{n-1} \end{bmatrix},$$

$$\mathbf{A} = \begin{bmatrix} a_0 \\ a_1 \\ \vdots \\ a_m \end{bmatrix},$$

$$\mathbf{B} = \begin{bmatrix} b_0 \\ b_1 \\ \vdots \\ b_{n-1} \end{bmatrix},$$

and

$$\mathbf{W} = \begin{bmatrix} H_1(i\omega_1)^{n} \\ H_2(i\omega_2)^{n} \\ \vdots \\ H_L(i\omega_L)^{n} \end{bmatrix}.$$

We want to minimize the sum of squared errors:

$$J = \sum_{j=1}^{L} e_j^* e_j = \mathbf{E}^H \mathbf{E},$$

where $^H$ denotes the Hermitian (complex conjugate transpose). The error function $J$ is thus a function of the coefficient vectors $\mathbf{A}$ and $\mathbf{B}$:

$$\begin{aligned}
J\left(\mathbf{A}, \mathbf{B}\right) &= \left[\mathbf{P}\mathbf{A} - \mathbf{T}\mathbf{B} - \mathbf{W}\right]^H\left[\mathbf{P}\mathbf{A} - \mathbf{T}\mathbf{B} - \mathbf{W}\right] = \\
&=\left[\mathbf{A}^\intercal\mathbf{P}^H - \mathbf{B}^\intercal\mathbf{T}^H - \mathbf{W}^H\right]\left[\mathbf{P}\mathbf{A} - \mathbf{T}\mathbf{B} - \mathbf{W}\right] = \\
&=\mathbf{A}^\intercal\mathbf{P}^H\mathbf{P}\mathbf{A} - \mathbf{A}^\intercal\mathbf{P}^H\mathbf{T}\mathbf{B} - \mathbf{A}^\intercal\mathbf{P}^H\mathbf{W} - \mathbf{B}^\intercal\mathbf{T}^H\mathbf{P}\mathbf{A} + \mathbf{B}^\intercal\mathbf{T}^H\mathbf{T}\mathbf{B} + \mathbf{B}^\intercal\mathbf{T}^H\mathbf{W} - \mathbf{W}^H\mathbf{P}\mathbf{A} + \mathbf{W}^H\mathbf{T}\mathbf{B} + \mathbf{W}^H\mathbf{W},
\end{aligned}$$

where we have applied the fact that the Hermitian of the real vectors $\mathbf{A}$ and $\mathbf{B}$ is simply their transpose. For complex values $z$, we know that $z + z^* = 2\mathrm{Re}(z)$, where $\mathrm{Re}$ denotes the real part of $z$. Also, for scalar products of vectors and matrices, we have:

$$\left[\mathbf{A}^\intercal \mathbf{P}^H \mathbf{T} \mathbf{B}\right]^H = \mathbf{B}^\intercal \mathbf{T}^H \mathbf{P} \mathbf{A}$$

Applying this to the paired terms:

$$\mathbf{A}^\intercal \mathbf{P}^H \mathbf{T} \mathbf{B} + \mathbf{B}^\intercal \mathbf{T}^H \mathbf{P} \mathbf{A} = 2\mathrm{Re}\left(\mathbf{A}^\intercal \mathbf{P}^H \mathbf{T} \mathbf{B}\right)$$

Similarly:

$$\mathbf{A}^\intercal \mathbf{P}^H \mathbf{W} + \mathbf{W}^H \mathbf{P}\mathbf{A} = 2\mathrm{Re}\left(\mathbf{A}^\intercal \mathbf{P}^H \mathbf{W}\right)$$
$$\mathbf{B}^\intercal \mathbf{T}^H \mathbf{W} + \mathbf{W}^T \mathbf{T} \mathbf{B} = 2\mathrm{Re}\left(\mathbf{B}^\intercal \mathbf{T}^H \mathbf{W}\right)$$

Substituting these into the expression for $J$:

$$J = \mathbf{A}^\intercal\mathbf{P}^H\mathbf{P}\mathbf{A} + \mathbf{B}^\intercal\mathbf{T}^H\mathbf{T}\mathbf{B} + \mathbf{W}^H\mathbf{W} - 2\mathrm{Re}\left(\mathbf{A}^\intercal\mathbf{P}^H\mathbf{T}\mathbf{B}\right) - 2\mathrm{Re}\left(\mathbf{A}^\intercal\mathbf{P}^H\mathbf{W}\right) - 2\mathrm{Re}\left(\mathbf{B}^\intercal \mathbf{T}^H \mathbf{W}\right).$$

Taking the derivatives of $J$ with respect to $\mathbf{A}$ and $\mathbf{B}$ and setting them to zero:

$$\frac{\partial J}{\partial \mathbf{A}} = 2\mathbf{P}^H\mathbf{P}\mathbf{A} - 2\mathrm{Re}\left(\mathbf{P}^H\mathbf{T}\mathbf{B}\right) - 2\mathrm{Re}\left(\mathbf{P}^H\mathbf{W}\right) = \mathbf{0}$$

$$\frac{\partial J}{\partial \mathbf{B}} = 2\mathbf{T}^H\mathbf{T}\mathbf{B} - 2\mathrm{Re}\left(\mathbf{T}^H\mathbf{P}\mathbf{A}\right) - 2\mathrm{Re}\left(\mathbf{T}^H\mathbf{W}\right) = \mathbf{0},$$

where again we have used the identity $\left[\mathbf{A}^\intercal \mathbf{P}^H \mathbf{T} \mathbf{B}\right]^H = \mathbf{B}^\intercal \mathbf{T}^H \mathbf{P} \mathbf{A}$.

These equations can be solved to find $\{a\}$ and $\{b\}$, giving us the coefficients of the rational fraction polynomials that best fit the measured FRF data.

Once the coefficients are determined, the poles of the system can be found by solving for the roots of the denominator polynomial:

$$\sum_{k=0}^{2n} b_k s^k = 0$$

From these poles, we can extract the natural frequencies and damping ratios as described earlier.

The residues, which are related to the mode shapes, can be determined by using the partial fraction expansion:

$$R_{j,k} = \frac{\sum_{k=0}^{m} a_k \lambda_r^k}{(\lambda_r - \lambda_r^*)\prod_{i=1,i\neq r}^{n}(\lambda_r - \lambda_i)(\lambda_r - \lambda_i^*)}$$

These residues can then be used to reconstruct the mode shapes of the system.

### Orthogonal Polynomials Implementation <a name="orthogonal-polynomials"></a>

One challenge with the standard RFP formulation is that it can lead to ill-conditioned numerical problems, especially for high-order polynomials. This is due to the dynamic range of the polynomial terms and the potentially ill-conditioned matrices $[P]^H[P]$ and $[T]^H[T]$.

To overcome these issues, Richardson and Formenti proposed using orthogonal polynomials. The idea is to replace the standard polynomial basis with orthogonal polynomials that satisfy specific orthogonality conditions. This significantly improves the numerical conditioning of the problem.

The FRF can be rewritten in terms of orthogonal polynomials:

$$H(\omega_i) = \frac{\sum_{k=0}^{m} c_k \phi_{i,k}}{\sum_{k=0}^{2n} d_k \theta_{i,k}}$$

Where $\phi_{i,k}$ and $\theta_{i,k}$ are the values of the orthogonal polynomials at frequency $\omega_i$, and $c_k$ and $d_k$ are the new coefficients to be determined.

The orthogonality conditions are:

$$\sum_{i=1}^{L} \phi_{i,k}^* \phi_{i,j} = \begin{cases} 0, & k \neq j \\ \delta_k, & k = j \end{cases}$$

$$\sum_{i=1}^{L} \theta_{i,k}^* |H_i|^2 \theta_{i,j} = \begin{cases} 0, & k \neq j \\ \epsilon_k, & k = j \end{cases}$$

Where $\delta_k$ and $\epsilon_k$ are scaling factors.

The Forsythe method is commonly used to generate these orthogonal polynomials. For complex polynomials, this method generates the polynomials recursively:

$$P_{i,-1} = 0$$
$$P_{i,0} = 1$$
$$P_{i,1} = (X_i - U_1)P_{i,0}$$
$$P_{i,2} = (X_i - U_2)P_{i,1} - V_1P_{i,0}$$
$$P_{i,k} = (X_i - U_k)P_{i,k-1} - V_{k-1}P_{i,k-2}$$

Where $X_i = i\omega_i$ and $U_k$ and $V_k$ are coefficients determined by the orthogonality conditions.

For real orthogonal polynomials used with complex data, a simplified recursion can be derived:

$$R_{i,-1}^+ = 0$$
$$R_{i,0}^+ = \sqrt{\frac{1}{2\sum_{i=1}^{L} q_i}}$$
$$S_{i,k}^+ = \omega_i R_{i,k-1}^+ - V_{k-1} R_{i,k-2}^+$$
$$R_{i,k}^+ = \frac{S_{i,k}^+}{\sqrt{\sum_{i=1}^{L} (S_{i,k}^+)^2 q_i}}$$

Where $q_i$ is the weighting function at frequency $\omega_i$. For the numerator polynomials, $q_i = 1$, while for the denominator polynomials, $q_i = |H_i|^2$.

With orthogonal polynomials, the error function becomes:

$$e_i = \frac{\sum_{k=0}^{m} c_k \phi_{i,k}}{\sum_{k=0}^{2n} d_k \theta_{i,k}} - H_i$$

Multiplying by the denominator:

$$e_i = \sum_{k=0}^{m} c_k \phi_{i,k} - H_i \sum_{k=0}^{2n} d_k \theta_{i,k}$$

In matrix form:

$$\{e\} = [P]\{c\} - [T]\{d\} - \{w\}$$

Where now $[P]$ and $[T]$ are matrices of orthogonal polynomials:

$$[P] = \begin{bmatrix} 
\phi_{1,0} & \phi_{1,1} & \ldots & \phi_{1,m} \\
\phi_{2,0} & \phi_{2,1} & \ldots & \phi_{2,m} \\
\vdots & \vdots & \ddots & \vdots \\
\phi_{L,0} & \phi_{L,1} & \ldots & \phi_{L,m}
\end{bmatrix}$$

$$[T] = \begin{bmatrix} 
H_1\theta_{1,0} & H_1\theta_{1,1} & \ldots & H_1\theta_{1,2n-1} \\
H_2\theta_{2,0} & H_2\theta_{2,1} & \ldots & H_2\theta_{2,2n-1} \\
\vdots & \vdots & \ddots & \vdots \\
H_L\theta_{L,0} & H_L\theta_{L,1} & \ldots & H_L\theta_{L,2n-1}
\end{bmatrix}$$

$$\{w\} = \begin{bmatrix} 
H_1\theta_{1,2n} \\
H_2\theta_{2,2n} \\
\vdots \\
H_L\theta_{L,2n}
\end{bmatrix}$$

Due to the orthogonality conditions, the matrices $[P]^H[P]$ and $[T]^H[T]$ become diagonal, which greatly simplifies the solution:

$$[P]^H[P] = [D_P]$$
$$[T]^H[T] = [D_T]$$

Where $[D_P]$ and $[D_T]$ are diagonal matrices.

The solution equations then become:

$$[D_P]\{c\} - [X]\{d\} = \{h\}$$
$$[X]^\intercal\{c\} + [D_T]\{d\} = \{0\}$$

Where $[X] = \text{Re}([P]^H[T])$ and $\{h\} = \text{Re}([P]^H\{w\})$.

These can be solved more efficiently:

$$\{d\} = -([D_T]^R)^{-1}\{h_0\}^R$$
$$\{c\} = [D_P]^{-1}(\{h\} - [X]\{d\})$$

Where $R$ denotes the real part.

Once the coefficients $\{c\}$ and $\{d\}$ are determined, they can be converted back to the ordinary polynomial coefficients $\{a\}$ and $\{b\}$ using the properties of the orthogonal polynomials.

The poles and residues can then be calculated as before to obtain the modal parameters and mode shapes.

### Global Parameter Estimation <a name="global-estimation"></a>

The standard RFP method described above is applied to a single FRF measurement. However, for a complex structure with multiple measurement points, we often have multiple FRF measurements. The Global Rational Fraction Polynomial (GRFP) method extends the RFP method to utilize multiple FRF measurements simultaneously.

The key idea in the GRFP method is that while the numerator polynomials vary for different measurement points (due to different mode shapes), the denominator polynomial is common to all measurement points since it represents the system's characteristic equation.

For a set of FRF measurements $H_{j,k}(\omega)$ where $j$ represents different response locations and $k$ represents different excitation locations, the GRFP method formulates the problem as:

$$H_{j,k}(\omega) = \frac{\sum_{r=0}^{m} a_{j,k,r} (i\omega)^r}{\sum_{r=0}^{2n} b_r (i\omega)^r}$$

Notice that the coefficients $a_{j,k,r}$ depend on both the response and excitation locations, while the coefficients $b_r$ are common to all measurements.

The error function for each measurement is:

$$e_{j,k,i} = \sum_{r=0}^{m} a_{j,k,r} (i\omega_i)^r - H_{j,k,i}\left(\sum_{r=0}^{2n-1} b_r (i\omega_i)^r + (i\omega_i)^{2n}\right)$$

And the total error to be minimized is:

$$J = \sum_{j=1}^{N_j} \sum_{k=1}^{N_k} \sum_{i=1}^{L} |e_{j,k,i}|^2$$

Where $N_j$ is the number of response locations, $N_k$ is the number of excitation locations, and $L$ is the number of frequency points.

This leads to a larger set of equations:

$$\begin{bmatrix} 
[D_{P,1}] & [0] & \ldots & [0] & [X_1] \\
[0] & [D_{P,2}] & \ldots & [0] & [X_2] \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
[0] & [0] & \ldots & [D_{P,N}] & [X_N] \\
[X_1]^\intercal & [X_2]^\intercal & \ldots & [X_N]^\intercal & [D_T]
\end{bmatrix} 
\begin{bmatrix}
\{c_1\} \\
\{c_2\} \\
\vdots \\
\{c_N\} \\
\{d\}
\end{bmatrix} = 
\begin{bmatrix}
\{h_1\} \\
\{h_2\} \\
\vdots \\
\{h_N\} \\
\{0\}
\end{bmatrix}$$

Where $N = N_j \times N_k$ is the total number of FRF measurements.

The solution can still be found efficiently by first solving for $\{d\}$:

$$\{d\} = \left(\sum_{j=1}^{N} [D_T]^R\right)^{-1} \left(\sum_{j=1}^{N} \{h_0\}^R\right)$$

And then finding each $\{c_j\}$:

$$\{c_j\} = [D_{P,j}]^{-1}(\{h_j\} - [X_j]\{d\})$$

Once the global denominator coefficients $\{d\}$ are determined, they can be converted to ordinary polynomial coefficients, and the poles can be calculated. The residues for each measurement point can then be determined, providing a consistent set of modal parameters and mode shapes for the entire structure.

For our 2-DOF system, applying the GRFP method would mean simultaneously analyzing the FRFs $H_{1,2}(\omega)$ and $H_{2,2}(\omega)$ to find a common set of poles (natural frequencies and damping ratios) and the corresponding residues (related to mode shapes).

The standard RFP method would analyze each FRF separately, potentially leading to different estimates of the poles for each measurement. The GRFP method enforces the physical constraint of common poles, leading to more consistent and accurate estimates of the modal parameters.

The benefits of the GRFP method become even more pronounced for more complex systems with many degrees of freedom and measurement points, as it leverages the additional information from multiple measurements to obtain a more robust and reliable identification of the system's dynamics.